# 00. Pozyskanie i wstępna kontrola danych

## Cel notebooka

Celem notebooka jest wczytanie zbioru Default of Credit Card Clients z repozytorium UCI Machine Learning Repository oraz przeprowadzenie wstępnej kontroli jego struktury i jakości.

Plan pracy:
1. Wczytanie ustawień konfigurujących
    -config.yaml
    -src/data_loader.py
2. Kontrola pliku źródłowego
    - lokazlizacja
    - poprawność formatu
3. Wczytanie danych
    - odczyt z xls do DataFrame
    - weryfikacja liczby kolumn i wierszy
    - sprawdzenie czy plik zawiera oczekiwane dane


Dane źródłowe pozostają niezmodyfikowane.

In [1]:
from pathlib import Path
import sys

current_path = Path.cwd().resolve()

PROJECT_ROOT = next(
    (p for p in [current_path, *current_path.parents]
    if (p/"src").is_dir()),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Folder src not found.")

#Add project root to Python import paths
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0,str(PROJECT_ROOT))

print("Project catalog:", PROJECT_ROOT)

Project catalog: C:\Users\justy\Documents\Studia podyplomowe\Praca dyplomowa\credit-default-calibration


In [2]:
import importlib
from src import data_loader as dl

# Reload module to include the latest changes
importlib.reload(dl)



<module 'src.data_loader' from 'C:\\Users\\justy\\Documents\\Studia podyplomowe\\Praca dyplomowa\\credit-default-calibration\\src\\data_loader.py'>

In [4]:
import pandas as pd
import numpy as np

from src import data_loader as dl

#load project configuration and set seed
CONFIG_PATH = PROJECT_ROOT / "config.yaml"
config = dl.load_config(CONFIG_PATH)
dl.set_seed(config["project"]["random_seed"])

#define project paths

RAW_DATA_PATH = dl.get_data_path(config, PROJECT_ROOT, "raw_dir")
PROCESSED_DATA_PATH = dl.get_data_path(config, PROJECT_ROOT, "processed_dir")

print("Project:", config["project"]["name"])
print("Seed:", config["project"]["random_seed"])
print("Raw data directory:", RAW_DATA_PATH)
print("Processed data directory:", PROCESSED_DATA_PATH)
print("Configuration loaded successfully.")

Project: credit-default-calibration
Seed: 42
Raw data directory: C:\Users\justy\Documents\Studia podyplomowe\Praca dyplomowa\credit-default-calibration\data\raw
Processed data directory: C:\Users\justy\Documents\Studia podyplomowe\Praca dyplomowa\credit-default-calibration\data\processed
Configuration loaded successfully.


2. Load the Excel file

In [5]:
excel_files = list(RAW_DATA_PATH.glob("*.xls*"))

if len(excel_files) != 1:
    raise FileNotFoundError(f"Expected one XLS file, while found {len(excel_files)}.")

DATA_FILE = excel_files[0]

#xlrd engine is used for reading older Excel files (.xls)

df = pd.read_excel(DATA_FILE, header=1, engine="xlrd") 
print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

Dataset loaded successfully!
Dataset shape: (30000, 25)


In [6]:
display(df.head())

df.info()

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 25 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   ID                          30000 non-null  int64
 1   LIMIT_BAL                   30000 non-null  int64
 2   SEX                         30000 non-null  int64
 3   EDUCATION                   30000 non-null  int64
 4   MARRIAGE                    30000 non-null  int64
 5   AGE                         30000 non-null  int64
 6   PAY_0                       30000 non-null  int64
 7   PAY_2                       30000 non-null  int64
 8   PAY_3                       30000 non-null  int64
 9   PAY_4                       30000 non-null  int64
 10  PAY_5                       30000 non-null  int64
 11  PAY_6                       30000 non-null  int64
 12  BILL_AMT1                   30000 non-null  int64
 13  BILL_AMT2                   30000 non-null  int64
 14  BILL_A

In [7]:
print("Number of columns:", len(df.columns))
print("\nColumn names:")
print(df.columns.tolist())

print("\nNumber of records:", len(df))

print("\nTarget variable candidates:")
print(df.columns[df.columns.str.lower().str.contains("default")].tolist())

print("\nNumber of duplicated records:", df.duplicated().sum())
print("\nFirst 5 records:")
print(df.head())

Number of columns: 25

Column names:
['ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'default payment next month']

Number of records: 30000

Target variable candidates:
['default payment next month']

Number of duplicated records: 0

First 5 records:
   ID  LIMIT_BAL  SEX  EDUCATION  MARRIAGE  AGE  PAY_0  PAY_2  PAY_3  PAY_4  \
0   1      20000    2          2         1   24      2      2     -1     -1   
1   2     120000    2          2         2   26     -1      2      0      0   
2   3      90000    2          2         2   34      0      0      0      0   
3   4      50000    2          2         1   37      0      0      0      0   
4   5      50000    1          2         1   57     -1      0     -1      0   

   ...  BILL_AMT4  BILL_AMT5  BILL_AMT6  PAY_AMT1  PAY_AMT

In [8]:
# Expected dataset structure based on the UCI dataset documentation

EXPECTED_ROWS = 30_000

EXPECTED_COLUMNS = [
    "ID",
    "LIMIT_BAL",
    "SEX",
    "EDUCATION",
    "MARRIAGE",
    "AGE",
    "PAY_0",
    "PAY_2",
    "PAY_3",
    "PAY_4",
    "PAY_5",
    "PAY_6",
    "BILL_AMT1",
    "BILL_AMT2",
    "BILL_AMT3",
    "BILL_AMT4",
    "BILL_AMT5",
    "BILL_AMT6",
    "PAY_AMT1",
    "PAY_AMT2",
    "PAY_AMT3",
    "PAY_AMT4",
    "PAY_AMT5",
    "PAY_AMT6",
    "default payment next month"
]


def validate_dataset_structure(df):
    """
    Validate the basic structure of the input dataset.

    Checks:
    - Expected number of records.
    - Expected column names.
    - Expected number of columns.
    - Expected column order.

    Raises ValueError if any structural requirement is not met.
    """

    errors = []

    # Check number of records
    if len(df) != EXPECTED_ROWS:
        errors.append(
            f"Expected {EXPECTED_ROWS} records, found {len(df)}."
        )

    # Check missing columns
    missing_columns = [
        col for col in EXPECTED_COLUMNS
        if col not in df.columns
    ]

    if missing_columns:
        errors.append(
            f"Missing required columns: {missing_columns}"
        )

    # Check unexpected columns
    unexpected_columns = [
        col for col in df.columns
        if col not in EXPECTED_COLUMNS
    ]

    if unexpected_columns:
        errors.append(
            f"Unexpected columns: {unexpected_columns}"
        )

    # Check column order
    if list(df.columns) != EXPECTED_COLUMNS:
        errors.append(
            "Column names or their order do not match "
            "the expected dataset structure."
        )

    # Stop execution if any structural requirement fails
    if errors:
        raise ValueError(
            "Dataset structure validation failed:\n- "
            + "\n- ".join(errors)
        )

    print("Dataset structure validation: PASSED")
    print(f"Number of records: {len(df)}")
    print(f"Number of columns: {len(df.columns)}")

In [9]:
validate_dataset_structure(df)

Dataset structure validation: PASSED
Number of records: 30000
Number of columns: 25


In [11]:
TARGET_COLUMN = "default payment next month"

def validate_id_and_target(df):
    """ Validate customer identifiers and target variable values. 
    
    Checks:
    - ID column doesn't contain duplicates or missing values.
    - Target column doesn't contain missing values and has only binary values (0 and 1).
    
    Raises ValueError if any validation fails.
    """

    errors = []

    missing_ids = df["ID"].isna().sum()

    if missing_ids > 0:
        errors.append(
            f"ID column contains {missing_ids} missing values."
        )

    duplicates_ids = df["ID"].duplicated().sum()

    if duplicates_ids > 0:
        errors.append(
            f"ID column contains {duplicates_ids} duplicate values."
        )

    missing_targets = df[TARGET_COLUMN].isna().sum()

    if missing_targets > 0:
        errors.append(
            f"Target column contains {missing_targets} missing values."
        )

    expected_target_values = {0, 1}
    actual_target_values = set(df[TARGET_COLUMN].dropna().unique())

    unexpected_target_values = actual_target_values - expected_target_values

    if unexpected_target_values:
        errors.append(
            f"Target column contains unexpected values: {unexpected_target_values}"
        )

    # Stop execution if any validation fails
    if errors:
        raise ValueError(
            "ID and target validation failed:\n- "
            + "\n- ".join(errors)
        )

    print("ID and target validation: PASSED")
    print(f"Unique customer IDs: {df['ID'].nunique()}")
    print(f"Target values: {sorted(actual_target_values)}")

    print("\nTarget distibution:")
    print(df[TARGET_COLUMN].value_counts().sort_index())

In [12]:
validate_id_and_target(df)

ID and target validation: PASSED
Unique customer IDs: 30000
Target values: [np.int64(0), np.int64(1)]

Target distibution:
default payment next month
0    23364
1     6636
Name: count, dtype: int64


In [ ]:
PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)

CSV_FILE = PROCESSED_DATA_PATH / "credit_default_data.csv"

#if the file already exists, it will be overwritten
df.to_csv(CSV_FILE, index=False)

print("Dataset saved successfully to:", CSV_FILE)
print("Saved dataset shape:", df.shape)

Dataset saved successfully to: C:\Users\justy\Documents\Studia podyplomowe\Praca dyplomowa\credit-default-calibration\data\processed\credit_default_data.csv
Saved dataset shape: (30000, 25)


In [14]:
df_check = pd.read_csv(CSV_FILE)

assert df_check.shape == df.shape, "Saved dataset shape does not match original dataset shape."
assert list(df_check.columns) == list(df.columns), "Saved dataset columns do not match original dataset columns."
assert df_check["ID"].is_unique, "Saved dataset ID column contains duplicates."
assert df_check[TARGET_COLUMN].isin([0,1]).all(), "Unexpected target values found in the saved dataset."

print("CSV validation: PASSED")
print("Shape:", df_check.shape)
print("File size:", round(CSV_FILE.stat().st_size / 1024, 2), "KB")
display(df_check.head())

CSV validation: PASSED
Shape: (30000, 25)
File size: 2829.31 KB


,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0
